# 07 — Ministry of Agriculture Prices | Bronze Ingestion (incremental, Auto Loader)

Capture every Ministry of Agriculture (MA) market price report that arrives in the raw archive, exactly once, as published.

| Item | Value |
|---|---|
| Archive | `/Volumes/rd_economic_intelligence/bronze/agriculture_raw/{retail_prices,wholesale_prices}/<yyyy>/<mm>/` (written by `07_MA_Agricultural_Prices_Download`) |
| Files | Retail (*Minorista*): legacy `.xls` with header row, 14 columns, one price column per market. Wholesale (*Mayorista*): `.xlsx` **without header row**, 8 columns, one price. Each report also comes as `;`-delimited CSV (Windows-1252), which rounds prices; Excel is authoritative, CSV is captured for an observational reconciliation. |
| Step 1 — discover | **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`) registers new files in `bronze.ma_agri_files` |
| Step 2 — capture | One Bronze row per source row; all cells kept as a JSON array of text (`row_json`); channel from the folder, format from the extension. Layout is interpreted in Silver. |
| History | Each report carries the full history since August 2017; Silver uses the newest snapshot per channel and format. |
| Target | `rd_economic_intelligence.bronze.ma_agri_prices_raw` |

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 xlrd==2.0.1 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import sys, uuid, json, csv, io, importlib
import openpyxl, xlrd
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
ROOT = f"/Volumes/{CATALOG}/bronze/agriculture_raw/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/agriculture_raw/_checkpoints/agri_price_files/"
FILES = f"{CATALOG}.bronze.ma_agri_files"
TARGET = f"{CATALOG}.bronze.ma_agri_prices_raw"
CHANNELS = {"retail_prices": "retail", "wholesale_prices": "wholesale"}
MIN_COLUMNS = {"retail": 14, "wholesale": 8}
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, ROOT, "*.{xls,xlsx,csv}", CHECKPOINT, FILES,
    comment="Bronze: register of Ministry of Agriculture price reports (retail and wholesale, Excel and CSV) found in the raw archive by Auto Loader.")
pending = ING.pending_files(spark, FILES, TARGET)
print(f"Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'].split('agriculture_raw/')[-1] for f in pending]}")

In [0]:
def as_text(v):
    if v is None:
        return None
    if isinstance(v, float):
        return str(int(v)) if v.is_integer() else repr(v)
    s = str(v)
    return s if s != "" else None

def excel_rows(path):
    if path.lower().endswith(".xls"):
        book = xlrd.open_workbook(path)
        sh = book.sheet_by_index(0)
        for r in range(sh.nrows):
            out = []
            for c in range(sh.ncols):
                cl = sh.cell(r, c)
                if cl.ctype in (xlrd.XL_CELL_EMPTY, xlrd.XL_CELL_BLANK):
                    out.append(None)
                elif cl.ctype == xlrd.XL_CELL_ERROR:
                    out.append(xlrd.error_text_from_code.get(cl.value, "#ERROR"))
                elif cl.ctype == xlrd.XL_CELL_DATE:
                    out.append(xlrd.xldate_as_datetime(cl.value, book.datemode).isoformat())
                else:
                    out.append(cl.value)
            yield out
    else:
        ws = openpyxl.load_workbook(path, read_only=True, data_only=True).worksheets[0]
        for r in ws.iter_rows(values_only=True):
            yield list(r)

def csv_rows(path):
    text = open(path, "rb").read().decode("cp1252", errors="replace")
    yield from csv.reader(io.StringIO(text), delimiter=";")

captured = {}
for f in pending:
    rel = f["_source_path"].split("agriculture_raw/")[-1]
    channel = CHANNELS[rel.split("/")[0]]
    fmt = "csv" if f["_source_file"].lower().endswith(".csv") else "excel"
    local = ING.local_path(f["_source_path"])
    rows = []
    for i, r in enumerate(excel_rows(local) if fmt == "excel" else csv_rows(local), start=1):
        vals = [as_text(v) for v in r]
        while vals and vals[-1] is None:
            vals.pop()
        if any(v is not None and v.strip() for v in vals):
            rows.append((channel, fmt, i, len(vals), json.dumps(vals, ensure_ascii=False)))
    widths = {}
    for r in rows:
        widths[r[3]] = widths.get(r[3], 0) + 1
    if max(widths, key=widths.get) < MIN_COLUMNS[channel]:
        raise ValueError(f"{rel}: structural contract failed, most rows have fewer than {MIN_COLUMNS[channel]} columns: {widths}")
    df = spark.createDataFrame(rows, "_channel string, _source_format string, _source_row_number int, cell_count int, row_json string")
    ING.lineage(df, f, BATCH_ID, INGESTION_TS).write.format("delta").mode("append").saveAsTable(TARGET)  # one atomic append per file
    captured[rel] = {"channel": channel, "format": fmt, "rows": len(rows), "cell_count_distribution": widths}
    print(rel, captured[rel])

if spark.catalog.tableExists(TARGET):
    spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: Ministry of Agriculture retail and wholesale market price reports as published (cells as JSON text arrays), one snapshot per archived file.'")
snapshots = spark.table(TARGET).select("_source_path").distinct().count() if spark.catalog.tableExists(TARGET) else 0
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
                                  "captured": captured, "snapshots_in_bronze": snapshots}, ensure_ascii=False))